# conrep — Analysis Notebook

**Simon Josef Durstewitz**
M.Sc. Social Data Science, University of Copenhagen

## Overview

This notebook implements the analysis pipeline accompanying Durstewitz (2026), *Concept Representations in Vector Space: Avenues for the Interdisciplinary Study of Meaning Variation*. The framework maps concept representations in vector space through textual traces and sentence embeddings, modeling pairwise representational distances to analyze meaning variation from three perspectives:

1. **Sharedness** — compute $S_b$ per concept, visualize across a concept set, and correlate with word-level properties.
2. **Locating Variation** — Mantel tests linking participant-level predictors to pairwise representational distance.
3. **Examining Deviation** — compare any target group's associations to a human reference distribution via permutation tests and variance decomposition.

The notebook is organized into two foundational sections followed by three analytical avenues.

The framework is demonstrated in this notebook on the Small World of Words English free association dataset (SWOW-EN2018; De Deyne et al., 2019), but accepts any dataset that can be structured as one row per participant per concept. Custom pre-built sentences can also be passed directly to the encoder. The notebook allows researchers to flexibly customize analyses, specifying their own concept sets and/or variables of interest.


In [ ]:
import pandas as pd
import numpy as np
import nltk
nltk.download('wordnet', quiet=True)

import conrep as fw
fw.set_style()


## 1. Data Input

Two modes are supported.

**Mode 1 — Associate data (SWOW format):**
Your dataset must have one row per participant per concept, with a fixed number of response columns (R1, R2, ...).
SWOW-EN2018 is the exemplary dataset. Any other free association dataset with the same structure works.
Synthetic sentences are constructed automatically per response following a uniform sentence template: *"Concept is associated with [R1], [R2], and [R3]."*

**Mode 2 — Full sentences:**
Bring your own pre-built sentences. Your DataFrame must have columns `participantID`, `cue`, `sentence`.
No preprocessing is performed; sentences are passed directly to the encoder.


In [ ]:
# Data mode — choose one, comment out the other
DATA_MODE = "associates"   # any fixed-response association dataset; sentences constructed automatically
# DATA_MODE = "sentences"  # pre-built sentences, no preprocessing

# Path to your data file
DATA_PATH = "path" #if you use the SWOW data set, specifiy the path to the SWOW data file here, e.g. "data/SWOW-EN.csv"

# Settings for DATA_MODE "associates"
RESPONSE_COLS    = ['R1', 'R2', 'R3']
MIN_RESPONSES    = 3
MIN_PARTICIPANTS = 75
NOUN_FILTER      = True
TEMPLATE         = "{cue} is associated with {responses}."
CUE_COL          = "cue"
PARTICIPANT_COL  = "participantID"

# Settings for DATA_MODE "sentences"
# SENTENCE_COL = "sentence"


In [ ]:
# Call functions 

if DATA_MODE == "associates":
    df_raw = pd.read_csv(DATA_PATH)
    print(f"Raw rows  : {len(df_raw):,}")
    print(f"Unique cues: {df_raw[CUE_COL].nunique():,}")

    df = fw.prepare_swow(
        df               = df_raw,
        response_cols    = RESPONSE_COLS,
        min_responses    = MIN_RESPONSES,
        cue_col          = CUE_COL,
        participant_col  = PARTICIPANT_COL,
        noun_filter      = NOUN_FILTER,
        min_participants = MIN_PARTICIPANTS,
        template         = TEMPLATE,
    )

elif DATA_MODE == "sentences":
    df_raw = pd.read_csv(DATA_PATH)
    df = fw.prepare_custom(
        df              = df_raw,
        cue_col         = CUE_COL,
        participant_col = PARTICIPANT_COL,
        sentence_col    = SENTENCE_COL,
    )

else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE!r}. Choose 'associates' or 'sentences'.")

print(df[['cue', 'sentence']].head(5).to_string(index=False))


## 2. Embedding

Sentences are encoded using a sentence encoder from the SentenceTransformers library.
Embeddings are cached to disk; subsequent runs load from cache automatically.
The encoder name is part of the cache filename, so switching encoders never loads stale data.


In [ ]:
# Choose a sentence encoder
# Any model from https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
ENCODER   = "all-mpnet-base-v2"
# ENCODER = "all-roberta-large-v1"
# ENCODER = "paraphrase-multilingual-mpnet-base-v2"

# Directory for caching embeddings
CACHE_DIR = "cache"


In [ ]:
encoder    = fw.load_encoder(ENCODER)
embeddings = fw.encode_concepts(
    df              = df,
    model           = encoder,
    cue_col         = CUE_COL,
    participant_col = PARTICIPANT_COL,
    cache_path      = CACHE_DIR,
    model_name      = ENCODER.replace('/', '_'),
)
print(f"Embeddings ready for {len(embeddings)} concepts.")


## 3. Avenue 1 — Comparing Sharedness

Two independent paths. Run either or both.

**Path 1:** define a concept list, compute $S_b$, visualize.
**Path 2:** compute $S_b$ across all retained concepts, specify a predictor and optional covariate, run the regression.


### Path 1 — Compare a concept list

In [ ]:
# Define the concepts you want to compare.
CONCEPTS = [
    'rain', 'forest', 'ice', 'snow',
    'justice', 'freedom', 'democracy', 'equality',
    'dog', 'chair', 'apple', 'car',
]


In [ ]:
df_sharedness = fw.compute_sharedness(embeddings, concepts=CONCEPTS)

# <= 25 concepts: ranked bar chart
# >  25 concepts: ranked table
if len(df_sharedness) <= 25:
    fig = fw.plot_sharedness(df_sharedness, save_path='sharedness.pdf')
else:
    print(df_sharedness[['concept', 'N', 'S_b']].to_string(index=False))


### Path 2 — Predict sharedness from word-level properties

In [ ]:
# Compute S_b across all retained concepts.
df_sharedness_all = fw.compute_sharedness(embeddings)
print(f"Concepts: {len(df_sharedness_all)}")
print(df_sharedness_all[['concept', 'N', 'S_b']].head(10).to_string(index=False))


In [ ]:
words = df_sharedness_all['concept'].tolist()

# WordNet-derived properties — computed automatically for all retained concepts
df_properties = fw.build_properties_table(words, {
    'synset_count':         fw.synset_count(words),
    'hypernym_depth':       fw.hypernym_depth(words),
    'hyponym_count':        fw.hyponym_count(words),
    'hypernym_count':       fw.hypernym_count(words),
    'lemma_count':          fw.lemma_count(words),
    'morphosemantic_links': fw.morphosemantic_links(words),
    'also_see_count':       fw.also_see_count(words),
    'topic_domain_count':   fw.topic_domain_count(words),
    'pos_diversity':        fw.pos_diversity(words),
})

# External norms — uncomment and set path to include
# df_properties['concreteness']       = fw.load_concreteness("/path/to/file")
# df_properties['valence']            = fw.load_valence("/path/to/file")
# df_properties['arousal']            = fw.load_arousal("/path/to/file")
# df_properties['dominance']          = fw.load_dominance("/path/to/file")
# df_properties['age_of_acquisition'] = fw.load_age_of_acquisition("/path/to/file")
# df_properties['word_frequency']     = fw.load_word_frequency("/path/to/file")
# df_properties['imageability']       = fw.load_imageability("/path/to/file")

print(df_properties.head())


In [ ]:
# Specify predictor and optional covariate.
# PREDICTOR must be a column name in df_properties —
# any of the WordNet-derived properties loaded above,
# or any external norm you uncommented.
# Set CONTROL to None to run a simple regression without partialling.
PREDICTOR       = 'hypernym_depth'   # example
PREDICTOR_LABEL = 'Hypernym depth'   # example
CONTROL         = 'synset_count'     # example — set to None to skip
CONTROL_LABEL   = 'Synset count'     # example


In [ ]:
df_reg = df_sharedness_all.set_index('concept').join(df_properties, how='inner').reset_index()
df_reg = df_reg.rename(columns={'index': 'concept'})

print(f"Concepts entering regression: {len(df_reg)}")

fig, result = fw.plot_partial_regression(
    df          = df_reg,
    y_col       = 'S_b',
    x_col       = PREDICTOR,
    control_col = CONTROL,
    weight_col  = 'N',
    ylabel      = r'Sharedness ($S_b$)',
    xlabel      = f'{PREDICTOR_LABEL} (residualized)' if CONTROL else PREDICTOR_LABEL,
    save_path   = 'sharedness_regression.pdf',
)

print(f"r = {result['r']:.3f}, p = {result['p']:.4f}, "
      f"95% CI [{result['r_ci_lo']:.3f}, {result['r_ci_hi']:.3f}]")


## 4. Avenue 2 — Locating Variation

Tests whether a continuous participant-level variable predicts pairwise
representational distance via permutation-based Mantel tests.

If your data contains city, country, or age columns, a set of example
predictors is computed automatically from participant metadata. Any
custom continuous pd.Series indexed by participantID can also be used.


In [ ]:
swow_predictors = fw.load_swow_predictors(df, participant_col=PARTICIPANT_COL)


In [ ]:
# Choose a predictor. The pairwise distance between participants is computed
# automatically: absolute difference for scalar predictors, great-circle
# distance for 'geographic_distance'.
#
# Example predictors from SWOW metadata:
#   'age'                        age in years
#   'abs_latitude'               absolute latitude of participant city in degrees
#                                (captures environmental distance)
#   'geographic_distance'        great-circle distance between participant cities (km)
#   'city_population'            population of participant city
#   'city_population_log'        log-transformed city population
#                                (recommended for skewed distributions)
#   'distance_to_capital'        distance from participant city to country capital (km)
#                                (proxy for urbanity)
#   'country_population'         population of participant's native country
#   'country_population_log'     log-transformed native country population
#                                (recommended for skewed distributions)
#   'country_gdp_per_capita'     GDP per capita in USD of participant's native country
#                                (proxy for economic development)
#   'country_gdp_per_capita_log' log-transformed GDP per capita
#                                (recommended for skewed distributions)
#
# Or supply any custom pd.Series indexed by participantID:
#   PREDICTOR_SERIES = my_dataframe.set_index('participantID')['my_variable']

PREDICTOR_NAME   = 'country_gdp_per_capita'                     # example
PREDICTOR_LABEL  = 'GDP per capita'  # example

PREDICTOR_SERIES = swow_predictors[PREDICTOR_NAME]


In [ ]:
# Define the concept set for the Mantel test.
# Keep this to a manageable size — the Mantel test constructs a full
# pairwise distance matrix per concept, which is computationally expensive.
CONCEPTS_MANTEL = [
    'poverty', 'wealth', 'justice', 'freedom', 'democracy', 'equality',
]


In [ ]:
df_dyads = fw.build_dyads(
    embeddings = embeddings,
    predictor  = PREDICTOR_SERIES,
    concepts   = CONCEPTS_MANTEL,
)

df_mantel = fw.mantel_test(
    df_dyads = df_dyads,
    concepts = CONCEPTS_MANTEL,
    n_perm   = 9999,
    tail     = 'upper',
)

if len(df_mantel) <= 25:
    fig = fw.plot_mantel_results(df_mantel, show_legend=True, save_path='mantel.pdf')
else:
    print(df_mantel[['cue', 'r', 'p_mantel', 'N_dyads']].to_string(index=False))


## 5. Avenue 3 — Examining Deviation

Tests whether a target group's concept representations deviate from a human reference
distribution. For each concept, cross-group pairwise distances (target-to-reference) are
compared against within-reference distances via a permutation test on the mean.

Two subsections run the same test on different kinds of target groups.


### 5a. Subpopulation within your dataset

Define a target group from participant metadata. For SWOW data, a set of pre-built
subgroups is available. For any dataset, a custom filter expression can be used.

Concepts where either target or reference N < 75 are dropped. A warning is printed
if either N < 20.


In [ ]:
# Pre-built SWOW subgroups — computed automatically from participant metadata.
# Uses the participant geo table attached by load_swow_predictors() (Avenue 2).
# If you skipped Avenue 2 or are using custom data without city/country columns,
# this falls back to age/gender/nativeLanguage/education only.
df_participants = swow_predictors.get('_df_participants') if 'swow_predictors' in dir() else None

swow_subgroups = fw.build_swow_subgroups(
    df              = df,
    df_participants = df_participants,
    participant_col = PARTICIPANT_COL,
)


In [ ]:
# Choose a target group.
#
# Pre-built SWOW subgroups (available after running the cell above):
#   'women'                females in the dataset
#   'men'                  males in the dataset
#   'under_25'             participants aged < 25
#   'above_65'             participants aged > 65
#   'native_english'       native English speakers
#   'non_native_english'   non-native English speakers
#   'university_degree'    Bachelor or Master degree (education >= 4)
#   'urban'                city population >= 300,000 (UN large-city threshold)
#   'rural'                city population < 300,000
#   'europe'               participants from European countries
#   'africa'               participants from African countries
#   'asia'                 participants from Asian countries
#   'americas'             participants from the Americas
#   'oceania'              participants from Oceania
#
# Or define a custom subgroup with a pandas query string:
#   target_ids = fw.build_custom_subgroup(df, "age > 65", participant_col=PARTICIPANT_COL)

TARGET_NAME = 'university_degree'   # example
target_ids  = swow_subgroups[TARGET_NAME]


In [ ]:
# Define concept set for the deviation test.
CONCEPTS_DEVIATION = [
    'university', 'capitalism', 'economy', 'job', 'research', 'science', 'education', 'school', 'knowledge', 'learning']


In [ ]:
df_deviation, dist_data_a = fw.run_deviation_test(
    embeddings           = embeddings,
    target_ids           = target_ids,
    concepts              = CONCEPTS_DEVIATION,
    n_perm               = 10000,
    min_n                = 75,
    warn_n               = 20,
    participant_col      = PARTICIPANT_COL,
    return_distributions = True,
)

print(df_deviation.to_string(index=False))

if len(df_deviation) <= 25 and len(df_deviation) > 0:
    fig = fw.plot_distribution_comparison(
        dist_data_a,
        group_a_label = 'Reference–reference',
        group_b_label = 'Target–reference',
        save_path     = 'deviation.pdf',
    )

In [ ]:
print(CONCEPTS_DEVIATION)

In [ ]:
import inspect
print("concept_seed" in inspect.getsource(fw.run_deviation_test))

### 5b. Subpopulation from external data

In principle, any external group whose textual responses can be encoded can serve as the target.

Within the field of machine behavior LLMs may present a subgroup of actors whose representational deviation to a human population has profound implications-

The following code presents a possibility to compare associations of an LLM to that of a human population, e.g., in the SWOW data set.

Associations are elicited and then preprocessed into synthetic sentences using the same sentence template as for the human participants *("Concept" is associated with [R1], [R2], and [R3])*, and then embedded with the same encoder.


In [ ]:
# Elicit associations from an LLM.
# The same sentence template is used as for human participants.
# Results are saved to llm_associations/ as a timestamped CSV.
# If you already have a CSV, load it directly and skip the collect call:
#   df_llm = pd.read_csv("llm_associations/associations_YYYYMMDD_HHMMSS.csv")
#
# Choose a provider. Only that provider's SDK needs to be installed
# (see requirements.txt for the optional install commands).
PROVIDER = "gemini"      # "gemini" | "openai" | "anthropic"

API_KEY   = "YOUR_API_KEY_HERE"
LLM_MODEL = "gemini-2.5-flash" # (example)
# LLM_MODEL = "gpt-4o-mini" (example)                 # if PROVIDER = "openai"
# LLM_MODEL = "claude-haiku-4-5-20251001" (example)    # if PROVIDER = "anthropic"

LLM_K    = 20      # association triples per concept
LLM_TEMP = 0.7

CONCEPTS_DEVIATION = [
    'justice', 'freedom', 'democracy', 'equality',
    'rain', 'forest', 'dog', 'chair',
]

df_llm = fw.collect_llm_associations(
    concepts    = CONCEPTS_DEVIATION,
    provider    = PROVIDER,
    api_key     = API_KEY,
    model_name  = LLM_MODEL,
    k           = LLM_K,
    temperature = LLM_TEMP,
)

In [ ]:
embeddings_llm = fw.encode_llm_associations(
    df_llm     = df_llm,
    model      = encoder,
    cache_path = CACHE_DIR,
)
print(f"LLM embeddings ready for {len(embeddings_llm)} concepts.")


In [ ]:
df_deviation_b = fw.run_deviation_test_external(
    embeddings_reference = embeddings,
    embeddings_target    = embeddings_llm,
    concepts             = CONCEPTS_DEVIATION,
    n_perm               = 10000,
    warn_n               = 20,
)

print(df_deviation_b.to_string(index=False))

if len(df_deviation_b) <= 25:
    fig = fw.plot_llm_distributions(
        fw.compare_llm_human(
            embeddings_human = embeddings,
            embeddings_llm   = embeddings_llm,
            concepts         = CONCEPTS_DEVIATION,
        ),
        save_path = 'deviation_b.pdf',
    )
